# 01 · Auditoria da base de Receita

**Esteira PASA:** `01 Receita` → `02 Despesa` → `03 Vidas` → `04 Consolidado` → Power BI / `relatorio_executivo.html`

## Objetivo do ficheiro bruto

`data/Receita.csv` traz a **receita de contraprestações** por segmento (`data_ref`, `Operadora`, `faixa_etaria`, `uf`, `cidade`, `Plano`). É o numerador do ROI e o denominador da sinistralidade calculados no caderno 04.

Este caderno **não transforma nem grava** dados: valida tipos, integridade e cobertura e documenta os achados que condicionam a análise consolidada.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")


def localizar_raiz_projeto(inicio: Path | None = None) -> Path:
    """Sobe a arvore de diretorios ate encontrar a pasta ``data/`` do projeto."""
    inicio = (inicio or Path.cwd()).resolve()
    for pasta in (inicio, *inicio.parents):
        if (pasta / "data").is_dir():
            return pasta
    raise FileNotFoundError("Pasta 'data/' nao encontrada. Execute o caderno a partir da pasta do projeto.")


PASTA_RAIZ = localizar_raiz_projeto()
CHAVES_SEGMENTO = ["data_ref", "Operadora", "faixa_etaria", "uf", "cidade", "Plano"]

CAMINHO_RECEITA = PASTA_RAIZ / "data" / "Receita.csv"
print(f"Fonte: {CAMINHO_RECEITA.relative_to(PASTA_RAIZ)}")

Fonte: data\Receita.csv


In [2]:
def validar_schema(df: pd.DataFrame, esperado: dict[str, str]) -> pd.DataFrame:
    """Compara colunas e tipos lidos com o esperado; interrompe se houver divergencia."""
    relatorio = pd.DataFrame({
        "coluna": list(esperado),
        "tipo_esperado": list(esperado.values()),
        "tipo_lido": [str(df[c].dtype) if c in df else "AUSENTE" for c in esperado],
    })
    relatorio["confere"] = [
        c in df and (
            pd.api.types.is_string_dtype(df[c]) if tipo == "texto"
            else pd.api.types.is_integer_dtype(df[c]) if tipo == "inteiro"
            else pd.api.types.is_float_dtype(df[c]) if tipo == "decimal"
            else False
        )
        for c, tipo in esperado.items()
    ]
    assert list(df.columns) == list(esperado), f"Colunas divergentes: {list(df.columns)}"
    assert relatorio["confere"].all(), "Ha colunas com tipo diferente do esperado."
    return relatorio


def diagnosticar_integridade(df: pd.DataFrame, chave: list[str] | None) -> pd.Series:
    """Nulos, duplicidades, cobertura temporal e unicidade da chave de segmento."""
    datas = pd.to_datetime(df["data_ref"], format="%Y-%m-%d", errors="coerce")
    return pd.Series({
        "linhas": len(df),
        "celulas_nulas": int(df.isna().sum().sum()),
        "linhas_duplicadas_integralmente": int(df.duplicated().sum()),
        "segmentos_repetidos": int(df.duplicated(subset=chave).sum()) if chave else "n/a",
        "datas_invalidas": int(datas.isna().sum()),
        "primeira_competencia": datas.min().strftime("%m/%Y"),
        "ultima_competencia": datas.max().strftime("%m/%Y"),
        "competencias": datas.nunique(),
        "operadoras": ", ".join(sorted(df["Operadora"].unique())),
    }, name="valor")

## 1. Carga e validação de tipos

O contrato esperado é: dimensões como texto e `receita` como decimal. `data_ref` chega como texto ISO (`AAAA-MM-DD`) e é validada como data na etapa seguinte.

In [3]:
df_receita = pd.read_csv(CAMINHO_RECEITA)

SCHEMA_RECEITA = {
    "Indicador": "texto", "data_ref": "texto", "faixa_etaria": "texto", "uf": "texto",
    "cidade": "texto", "Operadora": "texto", "Plano": "texto", "receita": "decimal",
}
display(validar_schema(df_receita, SCHEMA_RECEITA))
display(df_receita.head())

,coluna,tipo_esperado,tipo_lido,confere
0,Indicador,texto,str,True
1,data_ref,texto,str,True
2,faixa_etaria,texto,str,True
3,uf,texto,str,True
4,cidade,texto,str,True
5,Operadora,texto,str,True
6,Plano,texto,str,True
7,receita,decimal,float64,True


,Indicador,data_ref,faixa_etaria,uf,cidade,Operadora,Plano,receita
0,Receita,2026-07-01,59 e acima,MG,AIMORES,Operadora Y,A6C4C5FD81,"15,265.58"
1,Receita,2026-04-01,39 - 43,ES,VILA VELHA,Operadora Y,A6C4C5FD81,"18,171.84"
2,Receita,2026-04-01,54 - 58,MG,SANTA BARBARA,Operadora Y,C3376E2403,241.32
3,Receita,2026-04-01,59 e acima,PA,BELEM,Operadora Y,1BB752221F,"1,091.91"
4,Receita,2026-04-01,59 e acima,ES,VILA VELHA,Operadora Y,7BD3C453B0,"1,027,032.93"


## 2. Integridade estrutural

Nulos, linhas duplicadas, datas inválidas e unicidade da chave de segmento. Em Receita cada segmento deve aparecer **uma única vez** por competência.

In [4]:
integridade = diagnosticar_integridade(df_receita, CHAVES_SEGMENTO)
display(integridade.to_frame())

assert integridade["celulas_nulas"] == 0, "Ha valores nulos."
assert integridade["linhas_duplicadas_integralmente"] == 0, "Ha linhas duplicadas."
assert integridade["segmentos_repetidos"] == 0, "Ha segmentos repetidos."
assert integridade["datas_invalidas"] == 0, "Ha datas invalidas."
assert integridade["competencias"] == 12, "Esperadas 12 competencias (08/2025 a 07/2026)."
assert df_receita["Indicador"].eq("Receita").all()
assert df_receita["receita"].gt(0).all(), "Ha receita zerada ou negativa."

,valor
linhas,39405
celulas_nulas,0
linhas_duplicadas_integralmente,0
segmentos_repetidos,0
datas_invalidas,0
primeira_competencia,08/2025
ultima_competencia,07/2026
competencias,12
operadoras,Operadora Y


In [5]:
display(df_receita["receita"].describe().to_frame())

,receita
count,"39,405.00"
mean,"19,217.35"
std,"121,925.37"
min,3.04
25%,750.24
50%,"2,172.82"
75%,"5,737.83"
max,"2,605,069.04"


## 3. Cobertura por operadora e competência

Antes de calcular qualquer indicador, conferimos **quais operadoras têm receita** e se a série mensal está completa.

In [6]:
receita_mensal = (
    df_receita.pivot_table(index="data_ref", columns="Operadora", values="receita", aggfunc="sum")
    .reindex(columns=["Operadora X", "Operadora Y"])
)
display(receita_mensal)

cobertura = df_receita.groupby("Operadora").agg(
    linhas=("receita", "size"), competencias=("data_ref", "nunique"),
    planos=("Plano", "nunique"), receita_total=("receita", "sum"),
)
display(cobertura)

Operadora,Operadora X,Operadora Y
data_ref,,
2025-08-01,NaN,"59,873,873.41"
2025-09-01,NaN,"59,773,576.32"
2025-10-01,NaN,"64,658,258.13"
2025-11-01,NaN,"64,404,463.12"
2025-12-01,NaN,"64,248,113.20"
2026-01-01,NaN,"64,014,364.02"
2026-02-01,NaN,"63,793,674.24"
2026-03-01,NaN,"63,444,378.23"
2026-04-01,NaN,"63,324,972.38"


,linhas,competencias,planos,receita_total
Operadora,,,,
Operadora Y,39405,12,14,"757,259,505.38"


## 4. Achado crítico — ausência de receita da Operadora X

**Evidência:** as 39.405 linhas da base pertencem exclusivamente à **Operadora Y** (12 competências completas, 14 planos). A Operadora X — a maior carteira, cerca de 4,8× a Y em vidas — **não tem nenhuma linha de receita**.

**Risco:** imputar zero ou somar a despesa da X contra a receita da Y produz uma sinistralidade "total" sem sentido (acima de 100%) e um ROI negativo inexistente.

**Recomendação / tratamento adotado no caderno 04:** a receita da X é tratada como **ausente, nunca zero**. Sinistralidade e ROI da X ficam **indisponíveis** (`NaN`, `status_receita = "Ausente na base fornecida"`), e qualquer total consolidado considera apenas linhas com receita.

> A célula abaixo transforma o achado em barreira: se uma nova extração trouxer receita da X, o caderno falha e sinaliza que a narrativa precisa ser revista.

In [7]:
operadoras_com_receita = set(df_receita["Operadora"].unique())
print("Operadoras com receita:", sorted(operadoras_com_receita))
print("Linhas de receita da Operadora X:", int(df_receita["Operadora"].eq("Operadora X").sum()))

assert operadoras_com_receita == {"Operadora Y"}, "A base agora contem receita de outra operadora: revisar o caderno 04."
assert cobertura.at["Operadora Y", "competencias"] == 12

Operadoras com receita: ['Operadora Y']
Linhas de receita da Operadora X: 0


## 5. Conclusões da auditoria

| Verificação | Resultado |
|---|---|
| Colunas e tipos | Conforme o esperado (7 dimensões texto + `receita` decimal) |
| Nulos, duplicados, datas inválidas | Nenhum |
| Chave de segmento | Única — sem segmentos repetidos |
| Valores | Todos positivos |
| Cobertura | **Somente Operadora Y**, 12 competências (08/2025 a 07/2026) |

**Implicação para o caderno 04:** sinistralidade e ROI são calculados apenas para a Y; a X é reportada como *indisponível*. A conciliação receita × vidas por segmento (planos sem receita, vidas sem receita após jan/2026) é feita na seção 5 do caderno 04.